# 02 — Silver Transformation (fixed)

This notebook is designed for your current Azure Databricks / Unity Catalog setup.

It specifically fixes the error:

`[INVALID_EXTRACT_BASE_FIELD_TYPE] Can't extract a value from "event"... got "STRING"`

The notebook safely handles **both Bronze formats** you have used during this project:

1. A full USGS GeoJSON object containing a `features` array.
2. The GitHub tutorial format where Bronze stores only the list of earthquake feature objects.

It also handles `features` containing either `STRUCT` objects or JSON `STRING` values, so it never tries to access `event.properties...` until `event` has been verified as a Spark `STRUCT`.


In [0]:
# 1. Job parameter

dbutils.widgets.text("start_date", "2026-09-01")

start_date = dbutils.widgets.get("start_date").strip()

if not start_date:
    raise ValueError("start_date is empty. Pass a value such as 2026-09-01.")

print("start_date =", start_date)


In [0]:
# 2. Resolve your Unity Catalog volume paths

import os

# Optional override if the notebook's current catalog is not the catalog
# where you created the earthquake volumes.
dbutils.widgets.text("catalog_name", "")

catalog_override = dbutils.widgets.get("catalog_name").strip()

if catalog_override:
    CATALOG = catalog_override
else:
    CATALOG = spark.sql(
        "SELECT current_catalog() AS catalog"
    ).first()["catalog"]

SCHEMA = "earthquake"

BRONZE_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/bronze"
SILVER_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/silver"

print("Catalog     :", CATALOG)
print("Bronze root :", BRONZE_ROOT)
print("Silver root :", SILVER_ROOT)


In [0]:
# 3. Automatically find the Bronze file

# Format used in the version we built together:
candidate_1 = (
    f"{BRONZE_ROOT}/run_date={start_date}/earthquakes.geojson"
)

# Format used by the GitHub tutorial notebook:
candidate_2 = (
    f"{BRONZE_ROOT}/{start_date}_earthquake_data.json"
)

candidate_paths = [candidate_1, candidate_2]

BRONZE_FILE = None

for path in candidate_paths:
    if os.path.exists(path):
        BRONZE_FILE = path
        break

if BRONZE_FILE is None:
    raise FileNotFoundError(
        "No Bronze file was found. Checked:\n"
        + "\n".join(candidate_paths)
        + "\n\nRun the Bronze notebook for this start_date first."
    )

print("Bronze input found:")
print(BRONZE_FILE)


In [0]:
# 4. Read Bronze JSON

raw_df = (
    spark.read
    .option("multiline", "true")
    .json(BRONZE_FILE)
)

print("=== RAW BRONZE SCHEMA ===")
raw_df.printSchema()

print("Top-level columns:", raw_df.columns)


## Normalize the Bronze JSON

The GitHub notebook stores a JSON array of earthquake objects directly, while the version we built earlier can store a complete GeoJSON document containing a `features` array.

The next cells normalize **either format** into one column named `event`, and guarantee that `event` is a `STRUCT` before flattening it.


In [0]:
# 5. Imports and fallback event schema

from pyspark.sql.functions import (
    col,
    explode_outer,
    from_json,
    struct,
    lit,
    current_timestamp,
    timestamp_millis
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    ArrayType,
    TimestampType
)

# Used ONLY if an earthquake event is stored as a JSON string.
# Extra USGS fields not listed here are safely ignored.
event_schema = StructType([
    StructField("id", StringType(), True),

    StructField(
        "geometry",
        StructType([
            StructField("type", StringType(), True),
            StructField(
                "coordinates",
                ArrayType(DoubleType()),
                True
            )
        ]),
        True
    ),

    StructField(
        "properties",
        StructType([
            StructField("title", StringType(), True),
            StructField("place", StringType(), True),
            StructField("sig", LongType(), True),
            StructField("mag", DoubleType(), True),
            StructField("magType", StringType(), True),
            StructField("time", LongType(), True),
            StructField("updated", LongType(), True)
        ]),
        True
    ),

    StructField("type", StringType(), True)
])


In [0]:
# 6. Normalize both possible Bronze layouts

from pyspark.sql.types import StructType, StringType

if "features" in raw_df.columns:
    # Full GeoJSON:
    # { "type": "FeatureCollection", "features": [ ... ] }

    events_raw_df = raw_df.select(
        explode_outer("features").alias("event_raw")
    )

    raw_type = events_raw_df.schema["event_raw"].dataType

    print("Bronze format: full GeoJSON with features[]")
    print("event_raw datatype:", raw_type)

    if isinstance(raw_type, StructType):
        # Already parsed correctly by Spark.
        events_df = events_raw_df.select(
            col("event_raw").alias("event")
        )

    elif isinstance(raw_type, StringType):
        # JSON text -> parse into STRUCT.
        events_df = events_raw_df.select(
            from_json(
                col("event_raw"),
                event_schema
            ).alias("event")
        )

    else:
        raise TypeError(
            f"Unsupported features element type: {raw_type}"
        )

elif {"id", "geometry", "properties"}.issubset(set(raw_df.columns)):
    # GitHub tutorial Bronze:
    # [
    #   {"id": "...", "geometry": {...}, "properties": {...}},
    #   ...
    # ]

    print("Bronze format: direct array/list of earthquake objects")

    events_df = raw_df.select(
        struct(
            *[col(c) for c in raw_df.columns]
        ).alias("event")
    )

else:
    raise ValueError(
        "Unrecognized Bronze JSON structure. "
        f"Columns found: {raw_df.columns}"
    )

print("=== NORMALIZED EVENT SCHEMA ===")
events_df.printSchema()


In [0]:
# 7. Critical safety check

event_type = events_df.schema["event"].dataType

print("Final event datatype:", event_type)

if not isinstance(event_type, StructType):
    raise TypeError(
        "Silver cannot continue because event is not a STRUCT. "
        f"Actual datatype: {event_type}"
    )

print("OK: event is a STRUCT. Safe to flatten.")


## JSON → normal Silver table

The following transformation keeps the same main column names as the tutorial's Silver notebook (`id`, `longitude`, `latitude`, `elevation`, `title`, `place_description`, `sig`, `mag`, `magType`, `time`, `updated`) so the Gold logic remains compatible.


In [0]:
# 8. Flatten earthquake event data into a normal table

spark.conf.set("spark.sql.session.timeZone", "UTC")

silver_df = events_df.select(

    col("event.id")
        .cast("string")
        .alias("id"),

    col("event.geometry.coordinates")[0]
        .cast("double")
        .alias("longitude"),

    col("event.geometry.coordinates")[1]
        .cast("double")
        .alias("latitude"),

    col("event.geometry.coordinates")[2]
        .cast("double")
        .alias("elevation"),

    col("event.properties.title")
        .cast("string")
        .alias("title"),

    col("event.properties.place")
        .cast("string")
        .alias("place_description"),

    col("event.properties.sig")
        .cast("long")
        .alias("sig"),

    col("event.properties.mag")
        .cast("double")
        .alias("mag"),

    col("event.properties.magType")
        .cast("string")
        .alias("magType"),

    timestamp_millis(
        col("event.properties.time").cast("long")
    ).alias("time"),

    timestamp_millis(
        col("event.properties.updated").cast("long")
    ).alias("updated"),

    lit(start_date)
        .alias("run_date"),

    current_timestamp()
        .alias("silver_processed_at")
)

print("=== FLATTENED SILVER SCHEMA ===")
silver_df.printSchema()


In [0]:
# 9. Data-quality checks

# Do not replace missing coordinates with 0:
# (0, 0) is a real geographic location and would create fake data.

silver_df = (
    silver_df
    .filter(col("id").isNotNull())
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .filter(col("latitude").between(-90, 90))
    .filter(col("longitude").between(-180, 180))
    .dropDuplicates(["id"])
)

silver_count = silver_df.count()

print("Silver row count:", silver_count)

display(
    silver_df
    .orderBy(col("time").desc())
    .limit(20)
)


In [0]:
# 10. Save Silver as Parquet

# Parent folder remains compatible with the tutorial's Gold notebook:
# .../silver/earthquake_events_silver/
#
# Each date is written separately so rerunning the same day is idempotent.

SILVER_OUTPUT_PATH = (
    f"{SILVER_ROOT}/earthquake_events_silver/"
    f"run_date={start_date}"
)

(
    silver_df
    .write
    .mode("overwrite")
    .parquet(SILVER_OUTPUT_PATH)
)

print("Silver successfully written to:")
print(SILVER_OUTPUT_PATH)


In [0]:
# 11. Verify the written Silver data

silver_check = spark.read.parquet(
    SILVER_OUTPUT_PATH
)

print("Verified row count:", silver_check.count())
silver_check.printSchema()

display(silver_check.limit(20))


In [0]:
# 12. Verify files exist in the Silver volume

files_written = dbutils.fs.ls(SILVER_OUTPUT_PATH)

for file_info in files_written:
    print(file_info.path)


In [0]:
# 13. Return the Silver parent path to downstream orchestration (optional)

SILVER_PARENT_PATH = (
    f"{SILVER_ROOT}/earthquake_events_silver/"
)

print("Silver parent path:", SILVER_PARENT_PATH)

# Uncomment only if another notebook/task explicitly consumes this notebook output:
# dbutils.notebook.exit(SILVER_PARENT_PATH)
